# Report 2

In questo report analizziamo come realizzare una rete per la seguente PDE cinetica:
$$
    \partial_t u_t = \nabla_x\cdot\left(-fu+\frac12\nabla_x\cdot\left(\Sigma u\right)\right)
$$

Il modello di rete neurale è ispirato a quello del report 1:
$$
    u_\theta(x)=\texttt{SoftPlus}\left(N_\theta(x)+c(\theta)\right)\varphi(x)
$$
dove
- $\varphi$ è una gaussiana e $N_\theta$ è un modello di rete neurale Lipschitziano, questo fa sì che il risultato abbia code leggere a sufficienza per garantire che $u_\theta$ sia $L_1$ e positiva.
- Il regolarizzatore $c(\theta)$ fa sì che la rete abbia una specifica magnitudine, questo non garantisce che sia una densità, ma aiuta una successiva normalizzazione.

Per garantire un training valido è condizione necessaria avere:
- Differenziabilità della soluzione: lungo spazio e tempo vorremmo una soluzione Lipschitziana, in questo modo si può affrontare la dimensionalità temporale come semplice asse aggiunta.
- Condizioni di Integrabilità: la soluzione deve preservare la norma L1 e lo stato iniziale deve avere norma L1 finita.

## Differenziabilità
Riscriviamo il problema come segue:
$$
    \partial_t u = \sum_{ij} a_{ij}(t,x)\partial_{x_i,x_j}u + \sum_i b_i(t,x)\partial_{x_i}u + c(t,x)u
$$

Affinché la soluzione teorica sia Lipschitziana sia nello spazio che nel tempo è richiesto:
- **Uniforme parabolicità:** la matrice dei coefficienti di diffusione $A$ deve essere simmetrica e definita positiva ovunque (sia nello spazio che nel tempo). Inoltre esiste $\lambda>0$ tale che per ogni vettore $\xi$ vale: $$\sum_{ij} a_{ij}(t,x)\xi_i\xi_j \geq \lambda \left\|\xi\right\|^2$$
- **Regolarità dei coefficienti:** i coefficienti $a_{ij}, b_{i}, c$ devono essere limitati e differenziabili.
- **Condizione iniziale:** Lo stato iniziale deve essere limitato, integrabile e almeno Lipschitz.

Fenomeni da controllare:
- Spectral bias: succede quando i gradienti sono insolitamente alti, seppur la soluzione è Lipschitz. La rete dovrebbe aver fatica a lavorare sulle alte frequenze.

## Integrabilità
Poiché lo stato iniziale $u_0$ è una densità allora è, in particolare, integrabile.

La PDE, essendo scritta in forma di divergenza, conserva la massa, quindi conserva la norma $L_1$ di $u_0$.

## Stato Limite

Nei tempi lunghi la soluzione può convergere verso uno stato stazionario.

Per avere tale certezza, sarà sufficiente richiedere quanto segue:
- **Condizione di Lyapunov:** deve esistere una costante $c>0$ e una costante $M\geq0$ tali che, per $\left\|x\right\|$ sufficientemente grande vale: $$\langle b(x,t),x\rangle \leq -c\left\|x\right\|^2 + M$$
- **Sistema autonomo:** i coefficienti non dipendono dal tempo. In tal caso l'esistenza di un potenziale garantisce la convergenza allo stato limite. Se il potenziale è anche fortemente convesso c'è una convergenza molto rapida (quanto meno esponenziale)

## Metodo
Con questo setup si propone la seguente funzione di adattamento al problema:
$$
    u_\theta(x,t)=\texttt{SoftPlus}\left(M_\theta(x,t)\ell(t)+N_\theta(x)(1-\ell(t))\right)\varphi_t(x)
$$
dove:
- $M_\theta$ e $N_\theta$ sono reti neurali Lipschitziane e almeno 2-differenziabili, la prima è perturbative.
- $\ell$ è una funzione a valori in $[0,1]$ che decade monotonicamente a $0$ almeno come $1/x^2$.
- $\varphi_t$ serve sia come attention region che come controllo delle code spaziali, quindi decade esponenzialmente a $0$ 

Il metodo proposto verifica le seguenti proprietà:
- E' differenziabile dipendentemente alla differenziabilità dei submodelli.
- E' integrabile in ogni istante di tempo, poiché $\varphi_t$ ha code leggere per ogni $t$.
- Converge alla soluzione asintotica descritta da $N_\theta$

Per garantire la giusta magnitudine nel tempo si può inserire un parametro di regolarizzazione:
$$
    u_\theta(x,t)=\texttt{SoftPlus}\left(M_\theta(x,t)\ell(t)+N_\theta(x)+c(\theta)\right)\varphi_t(x)
$$
Questo parametro funziona per **Anchor Points** o **Anchor Distribution** per lo steady state il quale risolve l'equazione
$$
    0 = \nabla\cdot\left(-f_\infty u + \frac12\nabla\cdot\left(\Sigma_\infty u\right)\right)
$$
notoriamente lineare e quindi soggetta a morte neuronale ($u_\theta \equiv 0$).
$$
    c(\theta) = \texttt{SoftPlus}^{-1}\left(\frac{y^\star}{\varphi_\infty(x^\star)}\right) - N_\theta(x^\star)
$$

Per preservare la massa conviene lavorare sul suo andamento temporale:
$$
    0 = \partial_t \int u(x,t) dx = \int \partial_t u(x,t) dx
$$
Operativamente, considero un frame $t$ e un campionamento di punti che via monte carlo stimano $\int \partial_t u_\theta(x,t) dx$. Minimizzare questo integrale è diverso da minimizzare la norma della derivata temporale in quanto accetta anche valori negativi.

### Loss Function
La loss function quindi considera le seguenti funzioni:
- **Residuo della PDE**

